# 04 — Drizzle Combine with AstroDrizzle

**Purpose:** Run AstroDrizzle on the aligned exposures for each quasar to produce
a single geometrically corrected, cosmic-ray rejected mosaic.

**Inputs:** Aligned FLT files from `data/processed/<quasar>/aligned/`

**Outputs:** DRZ mosaics in `data/processed/<quasar>/drizzled/`

**Parameters:** All drizzle parameters are read from `config/wfc3_ir_drizzle_params.yaml`.
Do not hardcode parameters in this notebook — edit the config file instead.

**After running:** Use `/sync-params` to verify config and notebook stay in sync.

**Next step:** `05_inspect_outputs.ipynb`

## Imports

In [ ]:
import os
from pathlib import Path
import yaml
from drizzlepac import astrodrizzle

## Load parameters from config

In [ ]:
CONFIG_PATH = Path('../config/wfc3_ir_drizzle_params.yaml')
with open(CONFIG_PATH) as f:
    params = yaml.safe_load(f)

print('Drizzle parameters loaded:')
for k, v in params.items():
    print(f'  {k}: {v}')

## Define paths

In [ ]:
PROCESSED_DIR = Path('../data/processed')
quasar_dirs = sorted([d for d in PROCESSED_DIR.iterdir() if d.is_dir()])
print(f'Quasars to drizzle: {[d.name for d in quasar_dirs]}')

## Run AstroDrizzle per quasar

In [ ]:
for qdir in quasar_dirs:
    aligned_dir = qdir / 'aligned'
    drizzled_dir = qdir / 'drizzled'
    drizzled_dir.mkdir(parents=True, exist_ok=True)
    
    flt_files = sorted(aligned_dir.glob('*flt.fits'))
    if not flt_files:
        print(f'{qdir.name}: no aligned FLT files — skipping')
        continue
    
    output_name = str(drizzled_dir / f'{qdir.name}_drz')
    print(f'\nDrizzling {qdir.name} ({len(flt_files)} exposures) → {output_name}')
    
    astrodrizzle.AstroDrizzle(
        input=[str(f) for f in flt_files],
        output=output_name,
        final_scale=params['final_scale'],
        final_pixfrac=params['final_pixfrac'],
        final_kernel=params['final_kernel'],
        driz_sep_bits=params['driz_sep_bits'],
        final_bits=params['final_bits'],
        combine_type=params['combine_type'],
        skymethod=params['skymethod'],
        skysub=params['skysub'],
        final_wcs=params['final_wcs'],
        clean=params['clean']
    )

## Next steps

Run `/pipeline-status` to confirm DRZ files were created for all quasars.
Then open `05_inspect_outputs.ipynb` to review the results.